In [0]:
# Instead of os.environ.get(), use dbutils.secrets.get()
from pyspark.sql.dataframe import DataFrame

def create_connection():
    return spark.read.format("jdbc") \
    .option("url", f"jdbc:mysql://{dbutils.secrets.get('wheelie', 'MYSQL_HOST')}/{dbutils.secrets.get('wheelie', 'MYSQL_DB')}") \
    .option("user", dbutils.secrets.get('wheelie', 'MYSQL_USERNAME')) \
    .option("password", dbutils.secrets.get('wheelie', 'MYSQL_PASSWORD'))

# mode "overwrite" or "append"
# https://stackoverflow.com/questions/66340758/insert-or-update-a-delta-table-from-a-dataframe-in-pyspark
def write_dim(df: DataFrame, table_name: str):
    display(df)
    df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"wheelie.data_warehouse.{table_name}")

c = create_connection()

In [0]:
# Standardized dataframe loading: one per table, all with _df suffix
inventory_df = c.option("dbtable", "inventory").load()
car_df = c.option("dbtable", "car").load()
inventory_equipment_df = c.option("dbtable", "inventory_equipment").load()
equipment_df = c.option("dbtable", "equipment").load()
staff_df = c.option("dbtable", "staff").load()
city_df = c.option("dbtable", "city").load()
address_df = c.option("dbtable", "address").load()
store_df = c.option("dbtable", "store").load()
country_df = c.option("dbtable", "country").load()
customer_df = c.option("dbtable", "customer").load()

# Join address with city to get city/country info
address_with_city_df = address_df.join(city_df, address_df.city_id == city_df.city_id, "left")

# Join city with country for full location info
city_with_country_df = city_df.join(country_df, "country_id")
address_with_location_df = address_df.join(city_with_country_df, "city_id")
cust_addr_df = customer_df.join(address_with_location_df, "address_id")

# Store + address join
store_with_address_df = store_df.join(
    address_df,
    store_df.address_id == address_df.address_id,  # Standardowy FK JOIN
    "left"
).select(
    store_df["*"],
    address_df.address,
    address_df.address2,
    address_df.postal_code,
    address_df.city_id
)

# Store + address + city join
store_with_city_df = store_with_address_df.join(
    city_df,
    store_with_address_df.city_id == city_df.city_id,
    "left"
).select(
    store_with_address_df["*"],
    city_df.city,
    city_df.country_id
)


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS wheelie;
GRANT ALL PRIVILEGES ON CATALOG wheelie TO `mm151@st.amu.edu.pl`;
GRANT ALL PRIVILEGES ON CATALOG wheelie TO `sebpie2@st.amu.edu.pl`;
GRANT ALL PRIVILEGES ON CATALOG wheelie TO `tr32342@st.amu.edu.pl`;
GRANT ALL PRIVILEGES ON CATALOG wheelie TO `a662d958-d69f-42df-b30c-66cb1c96944e`;
CREATE SCHEMA IF NOT EXISTS wheelie.data_warehouse;

In [0]:
# =====================================================
# DIM_DATE - PEŁNE ROZWIĄZANIE Z EXPLODE(SEQUENCE)
# NAJLEPSZA, NAJNIŻSZA, NAJSTABLINIEJSZA WERSJA
# =====================================================

import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from datetime import date
import hashlib

# =====================================================
# 1. KONFIGURACJA (dostosuj zakres dat)
# =====================================================
START_DATE = date(2000, 1, 1)   # Początek kalendarza
END_DATE = date(2027, 12, 31)   # Koniec kalendarza

# =====================================================
# 2. GENEROWANIE SEKWENCJI DAT - NAJLEPSZY SPOSÓB
# =====================================================
dates_df = spark.range(1).select(
    #  explode(sequence) - najszybsze i najprostsze!
    F.explode(
        F.sequence(
            F.lit(START_DATE),
            F.lit(END_DATE)
        )
    ).alias('date')
)

# =====================================================
# 3. TRANSFORMACJA DO DIM_DATE (wszystkie kolumny ze specyfikacji)
# =====================================================
dim_date = dates_df.select(
    # Surrogate key (unikalny ID)
    F.monotonically_increasing_id().alias('ID'),

    # Kluczowa kolumna - data
    F.col('date'),

    # Dzień tygodnia (ISO 8601: 1=Monday, 7=Sunday)
    F.dayofweek('date').alias('day_of_week'),
    F.when(F.dayofweek('date') == 1, 'Monday'   ).when(F.dayofweek('date') == 2, 'Tuesday'  ).when(F.dayofweek('date') == 3, 'Wednesday').when(F.dayofweek('date') == 4, 'Thursday').when(F.dayofweek('date') == 5, 'Friday'   ).when(F.dayofweek('date') == 6, 'Saturday').otherwise('Sunday').alias('day_of_week_name'),

    # Dzień/miesiąc/tydzień/rok
    F.dayofmonth('date').alias('day_of_month'),
    F.weekofyear('date').alias('week_of_year'),
    F.month('date').alias('month'),

    # Nazwy miesięcy
    F.when(F.month('date') ==  1, 'January' ).when(F.month('date') ==  2, 'February').when(F.month('date') ==  3, 'March'   ).when(F.month('date') ==  4, 'April'   ).when(F.month('date') ==  5, 'May'     ).when(F.month('date') ==  6, 'June'    ).when(F.month('date') ==  7, 'July'    ).when(F.month('date') ==  8, 'August'  ).when(F.month('date') ==  9, 'September').when(F.month('date') == 10, 'October'  ).when(F.month('date') == 11, 'November').otherwise('December').alias('month_name'),

    # Kwartal i rok
    F.quarter('date').alias('quarter'),
    F.year('date').alias('year'),

    # Flagi biznesowe
    (F.dayofweek('date').isin(6, 7)).alias('is_weekend'),

    # Flagi okresów COVID (zgodne ze specyfikacją)
    (F.col('date') < F.lit(date(2020, 3, 1))).alias('is_pre_covid'),
    (F.col('date').between(F.lit(date(2020, 3, 1)), F.lit(date(2022, 6, 30)))).alias('is_covid'),
    (F.col('date') > F.lit(date(2022, 6, 30))).alias('is_post_covid')
)

# =====================================================
# 4. FUNKCJA HASH (ze wszystkich kolumn oprócz ID)
# =====================================================
def generate_row_hash(df: DataFrame) -> DataFrame:
    """Generuje SHA256 hash ze wszystkich kolumn (bez ID)"""
    content_cols = [c for c in df.columns if c != 'ID']
    hash_input = F.concat_ws(
        '|',
        *[F.coalesce(F.col(c).cast('string'), F.lit('NULL')) for c in content_cols]
    )
    return df.withColumn('row_hash', F.sha2(hash_input, 256))

# =====================================================
# 5. FINALNA TABELA Z HASHEM
# =====================================================
dim_date_final = generate_row_hash(dim_date)

# =====================================================
# 6. REJESTRACJA TEMP VIEW (dla SQL queries)
# =====================================================
dim_date_final = dim_date_final.withColumnRenamed("row_hash", "date_key").drop("ID")
dim_date_final.createOrReplaceTempView('dim_date')

# =====================================================
# 7. ZAPIS DO DELTA TABLE (domyślny schemat)
# =====================================================
write_dim(dim_date_final, "dim_date")

# =====================================================
# GOTOWE! Tabela dim_date jest dostępna:
# =====================================================
# SQL: SELECT * FROM dim_date LIMIT 10
# Python: spark.table('dim_date').show()

#display(dim_date_final.select(
#    'ID', 'date', 'year', 'quarter', 'month', 'month_name',
#    'day_of_week_name', 'is_weekend', 'is_covid', 'row_hash'
#).limit(1000))  # limit(1000) dla wydajności

In [0]:
#STAFF

from pyspark.sql.functions import xxhash64, col

# Kimball hierarchy bridge for staff: staff_id, manager_id

from pyspark.sql import Window
import pyspark.sql.functions as F
# Add staff_key, staff_manager_key, staff_first_name, staff_last_name columns
staff_hierarchy = (
    staff_df
    .select(
        F.col("staff_id"),
        F.col("manager_id"),
        xxhash64(F.col("staff_id")).alias("staff_key"),
        xxhash64(F.col("manager_id")).alias("staff_manager_key"),
        F.col("first_name").alias("staff_first_name"),
        F.col("last_name").alias("staff_last_name")
    )
)

# Iteratively build hierarchy paths up to max_depth
max_depth = 10
hierarchy = staff_hierarchy.withColumn("path", F.array(F.col("staff_key"))) \
    .withColumn("level_path", F.array(F.lit(0))) \
    .withColumn("manager_keys", F.array(F.col("staff_manager_key")))

for i in range(1, max_depth + 1):
    hierarchy = hierarchy.join(
        staff_hierarchy.select(
            F.col("staff_key").alias(f"mgr_key_{i}"),
            F.col("staff_manager_key").alias(f"mgr_manager_key_{i}")
        ),
        hierarchy.manager_keys[i-1] == F.col(f"mgr_key_{i}"),
        "left"
    ).withColumn(
        "manager_keys", F.expr(f"concat(manager_keys, array({f'mgr_manager_key_{i}'}))")
    ).withColumn(
        "path", F.expr(f"concat(path, array({f'mgr_key_{i}'}))")
    ).withColumn(
        "level_path", F.expr(f"concat(level_path, array({i}))")
    )

# Flatten hierarchy: staff_key, staff_manager_key, level
bridge_df = hierarchy.select(
    F.col("staff_key"),
    F.col("staff_first_name"),
    F.col("staff_last_name"),
    F.posexplode(F.col("path")).alias("level", "staff_manager_key")
).filter(F.col("staff_manager_key").isNotNull())

# Join to get staff_manager names
bridge_df = bridge_df.join(
    staff_df.select(
        xxhash64(F.col("staff_id")).alias("staff_manager_key"),
        F.col("first_name").alias("staff_manager_first_name"),
        F.col("last_name").alias("staff_manager_last_name")
    ),
    "staff_manager_key",
    "left"
)

write_dim(bridge_df, "staff_hierarchy_bridge")

# Join staff with address_with_city_df and store_df
# Use explicit join conditions to avoid ambiguity

# Use bridge_df to get manager's first and last name for each staff
dim_staff = staff_df \
    .join(address_with_location_df, staff_df.address_id == address_with_location_df.address_id, "left") \
    .join(store_df, staff_df.store_id == store_df.store_id, "left") \
    .join(
        bridge_df.filter(col("level") == 1)
        .select("staff_key", "staff_manager_first_name", "staff_manager_last_name"),
        xxhash64(col("staff_id")) == bridge_df["staff_key"],
        "left"
    ) \
    .withColumn("staff_last_name", col("last_name")) \
    .withColumn("staff_first_name", col("first_name")) \
    .withColumn("staff_email", col("email")) \
    .select(
        "staff_key",
        "staff_id",
        store_df.store_id,
        staff_df.address_id,
        "staff_first_name",
        "staff_last_name",
        "staff_email",
        "hired_date",
        address_with_location_df.city.alias("staff_city"),
        address_with_location_df.country.alias("staff_country"),
        "staff_manager_first_name",
        "staff_manager_last_name"
    )

write_dim(dim_staff, "dim_staff")

In [0]:
# Import wymaganych funkcji Spark SQL do pracy z hashami i konkatenacją
from pyspark.sql.functions import sha2, concat_ws, col, lit
import pyspark.sql.functions as F  # Alias F dla pyspark.sql.functions

# 7. TRZECI JOIN: + country (po country_id z city)
dim_store_raw = store_with_city_df.join(
    country_df,
    store_with_city_df.country_id == country_df.country_id,  # city.country_id -> country.country_id
    "left"
).select(
    store_with_city_df.store_id,          # Business key (z source store.store_id)
    store_with_city_df.store_manager_id,  # Manager sklepu (staff_id)
    store_with_city_df.address_id,        # FK do address
    store_with_city_df.address,           # Denormalizowany adres 1
    store_with_city_df.address2,          # Denormalizowany adres 2
    store_with_city_df.city.alias("city"), # Nazwa miasta (alias żeby uniknąć kolizji)
    country_df.country,                # Nazwa kraju (USA, Canada)
    store_with_city_df.postal_code,       # Kod pocztowy
    store_with_city_df.last_update        # Timestamp ostatniej aktualizacji
)

dim_store_raw = dim_store_raw.join(
    staff_df,
    dim_store_raw.store_manager_id == staff_df.staff_id,
    "left"
).select(
    dim_store_raw["*"],
    staff_df.first_name.alias("store_manager_first_name"),
    staff_df.last_name.alias("store_manager_last_name"),
)

# 8. Tworzenie hash_key (surrogate key dla SCD Type 2)
# Lista wszystkich kolumn biznesowych do hashowania (bez last_update/ID)
business_cols = ["store_id", "store_manager_id", "address_id", "address", "address2", "city", "country", "postal_code"]
cols_str = [col(c).cast("string") for c in business_cols]  # Rzutuj wszystko na string

# Konkatenacja wszystkich kolumn z separatorem "||" + SHA2-256 hash
hash_key = sha2(concat_ws(lit("||"), *cols_str), 256)  # 256 bitów = 64 hex znaki
dim_store = dim_store_raw \
    .withColumn("hash_key", hash_key) \
    .withColumn("store_key", F.xxhash64(col("store_id")))  # Surrogate PK z hash store_id


# 10. Finalna kolejność kolumn (zgodna z dim_store schema)
dim_store = dim_store.select(
    "store_key",                    # Surrogate PK (1,2,3...)
    "store_id",              # Business key z source
    "address_id",            # FK do address
    "address",               # Denormalizowany
    "address2",              # Denormalizowany
    "city",                  # NOT NULL (z city.city)
    "country",               # NOT NULL (z country.country)
    "postal_code",           # Z address
    "store_manager_first_name", # Manager (staff_id)
    "store_manager_last_name",  # Manager (staff_id)
    "last_update",           # Audit field
    "hash_key"               # Nowy - do SCD Type 2
)

write_dim(dim_store, "dim_store")
#dim_store.show(truncate=False)
#dim_store.printSchema()

In [0]:
from pyspark.sql.functions import xxhash64, col, collect_set, concat_ws, xxhash64, array_sort

equipment_groups = inventory_equipment_df.groupBy("inventory_id") \
    .agg(array_sort(collect_set("equipment_id")).alias("equipments_array")) \
    .withColumn("equipments", concat_ws(",", col("equipments_array"))) \
    .withColumn("equipment_group_key", xxhash64(col("equipments"))) \
    .select("equipment_group_key", "equipments") \
    .distinct()

equipment_groups_normalized = equipment_groups \
    .withColumn("equipment_key_raw", F.explode(F.split(col("equipments"), ","))) \
    .withColumn("equipment_key", xxhash64(col("equipment_key_raw"))) \
    .withColumn("equipment_group_bridge_key", xxhash64(concat_ws("", col("equipment_group_key"), col("equipment_key")))) \
    .select(
        "equipment_group_bridge_key",
        "equipment_group_key",
        "equipment_key"
    )

# todo: drop column?
equipment_group_bridge = inventory_equipment_df.groupBy("inventory_id") \
    .agg(array_sort(collect_set("equipment_id")).alias("equipments_array")) \
    .withColumn("car_key", xxhash64(col("inventory_id"))) \
    .withColumn("equipments", concat_ws(",", col("equipments_array"))) \
    .withColumn("equipment_group_key", xxhash64(col("equipments"))) \
    .select(
        "car_key",
        "equipment_group_key",
    ).distinct()

dim_equipment = equipment_df \
    .withColumn("equipment_key", xxhash64(col("equipment_id"))) \
    .select(
        "equipment_key",
        "equipment_id",
        "name",
        "type",
        "version",
    )


inventory_df = inventory_df.withColumn(
    "fuel_type",
    F.when(F.lower(col("fuel_type")) == "diesle", "Diesel")
     .when(F.lower(col("fuel_type")) == "petol", "Petrol")
     .otherwise(col("fuel_type"))
)

dim_car = inventory_df.alias("inv").join(
    car_df.alias("car"),
    col("inv.car_id") == col("car.car_id"),
    "left"
).withColumn(
    "car_key", xxhash64(col("inv.inventory_id"))
).select(
    col("car_key"),
    col("car.car_id"),
    col("inv.inventory_id"),

    col("car.producer"),
    col("car.model"),
    col("car.rental_rate"),
    col("inv.production_year"),
    col("inv.fuel_type"),

    col("inv.license_plates"),
    col("inv.purchase_price"),
    col("inv.sell_price"),
    col("inv.store_id"),
    col("inv.last_update"),
)

write_dim(dim_car, "dim_car")
write_dim(dim_equipment, "dim_equipment")
write_dim(equipment_groups_normalized, "equipment_groups")
write_dim(equipment_group_bridge, "equipment_group_bridge")
display(equipment_group_bridge.select("equipment_group_key").distinct())
display(dim_car.select("fuel_type").distinct())

In [0]:
dim_customer = cust_addr_df \
    .withColumn("customer_key", xxhash64(col("customer_id"))) \
    .withColumn("customer_first_name", col("first_name")) \
    .withColumn("customer_last_name", col("last_name")) \
    .withColumn("customer_email", col("email")) \
    .withColumn("customer_city", col("city")) \
    .withColumn("customer_country", col("country")) \
    .select(
        "customer_key",
        "customer_id",
        "address_id",
        "customer_first_name",
        "customer_last_name",
        "customer_email",
        "birth_date",
        "customer_city",
        "customer_country",
        # TODO: SCD2
    )

write_dim(dim_customer, "dim_customer")

In [0]:
%sql
-- SELECT explode(sequence(<START>, <STOP>))